In [59]:
from z3 import *

In [60]:
x = Int('x')
y = Int('y')
solve(x > 2, y < 10, x + 2*y == 7)

[y = 0, x = 7]


In [65]:
x = Int('x')
y = Int('y')
print (simplify(x + y + 2*x + 3))
print (simplify(x < y + x + 2))
print (simplify(And(x + 1 >= 3, x**2 + x**2 + y**2 + 2 >= 5)))

3 + 3*x + y
Not(y <= -2)
And(x >= 2, 2*x**2 + y**2 >= 3)


In [66]:
x = Int('x')
y = Int('y')
print (x**2 + y**2 >= 1)
set_option(html_mode=False)
print (x**2 + y**2 >= 1)

x**2 + y**2 >= 1
x**2 + y**2 >= 1


In [67]:
x = Int('x')
y = Int('y')
n = x + y >= 3
print ("num args: ", n.num_args())
print ("children: ", n.children())
print ("1st child:", n.arg(0))
print ("2nd child:", n.arg(1))
print ("operator: ", n.decl())
print ("op name:  ", n.decl().name())

num args:  2
children:  [x + y, 3]
1st child: x + y
2nd child: 3
operator:  >=
op name:   >=


In [68]:
x = Real('x')
solve(x > 4, x < 0)

no solution


In [1]:
import argparse
from concurrent.futures import ProcessPoolExecutor
import json
import math
import shutil
import time
from pathlib import Path
from z3 import *
from util.KPathFinding2 import compute_k_paths

In [2]:
DEFAULT_INPUT_FILE = "D:\\masters\\project_work\\test_code\\input\\job_stress_test\\100_8.json"

In [3]:
def load_input(input_path):
    with open(input_path, "r") as f:
        return json.load(f)


In [4]:
input_file = None
data = {}

jobs_data = []
messages_data = []
platform_nodes = []
app_deadline = 0

endsystems = []
switches = []
all_nodes = []

num_endsystems = 0
num_switches = 0
num_nodes = 0

node_to_idx = {}
idx_to_node = {}
es_real_to_esidx = {}

adj = []
undirected_links = set()
path_data = {}
num_jobs = 0
num_msgs = 0
node_speed_factors = {}

In [7]:
data = load_input("./input/job_stress_test/53_8.json")

In [8]:
jobs_data      = data["application"]["jobs"]
messages_data  = data["application"]["messages"]
platform_nodes = data["platform"]["nodes"]
app_deadline   = data["application"]["deadline"]

In [9]:
endsystems = sorted([n["id"] for n in platform_nodes if not n["is_router"]])
switches   = sorted([n["id"] for n in platform_nodes if     n["is_router"]])
all_nodes  = endsystems + switches

In [10]:
node_to_idx      = {real_id: idx for idx, real_id in enumerate(all_nodes)}
idx_to_node      = {idx: real_id for real_id, idx  in node_to_idx.items()}
es_real_to_esidx = {real_id: i   for i, real_id    in enumerate(endsystems)}

In [11]:
print(es_real_to_esidx)

{1: 0, 2: 1, 3: 2, 7: 3, 11: 4, 15: 5, 16: 6, 17: 7}


In [12]:
for i, job in enumerate(jobs_data):
    allowed = [es_real_to_esidx[rid] for rid in job["can_run_on"] if rid in es_real_to_esidx]
    print(allowed)

[0]
[4, 7]
[3, 0, 1]
[2, 4, 7]
[5, 3]
[1]
[6, 5, 3]
[0, 1]
[7]
[3, 0]
[4, 7, 6]
[5, 3, 0]
[2, 4]
[6]
[1, 2, 4]
[7, 6]
[0]
[4, 7]
[3, 0, 1]
[2, 4, 7]
[5, 3]
[1]
[6, 5, 3]
[0, 1]
[7]
[3, 0]
[4, 7, 6]
[5, 3, 0]
[2, 4]
[6]
[1, 2, 4]
[7, 6]
[0]
[4, 7]
[3, 0, 1]
[2, 4, 7]
[5, 3]
[1]
[6, 5, 3]
[0, 1]
[7]
[3, 0]
[4, 7, 6]
[5, 3, 0]
[2, 4]
[6]
[1, 2, 4]
[7, 6]
[0]
[4, 7]
[3, 0, 1]
[2, 4, 7]
[5, 3]


In [13]:
print(jobs_data[0])

{'id': 0, 'wcet_fullspeed': 69, 'mcet': 0, 'processing_times': [69], 'can_run_on': [1]}


In [14]:
def job_duration_options(job):
    return {
        es_real_to_esidx[node_id]: duration
        for node_id, duration in zip(job["can_run_on"], job["processing_times"])
        if node_id in es_real_to_esidx
    }

In [ ]:
for job in jobs_data:
    options = job_duration_options(job)
    print(options)
    duration = job["wcet_fullspeed"]
    for es_idx, processing_time in reversed(list(options.items())):
        print(es_idx, processing_time, duration)

{0: 69}
0 69 69
{4: 54, 7: 108}
7 108 54
4 54 54
{3: 130, 0: 65, 1: 130}
1 130 65
0 65 65
3 130 65
{2: 74, 4: 49, 7: 98}
7 98 49
4 49 49
2 74 49
{5: 54, 3: 108}
3 108 54
5 54 54
{1: 122}
1 122 61
{6: 98, 5: 65, 3: 130}
3 130 65
5 65 65
6 98 65
{0: 39, 1: 78}
1 78 39
0 39 39
{7: 82}
7 82 41
{3: 96, 0: 48}
0 48 48
3 96 48
{4: 58, 7: 116, 6: 87}
6 87 58
7 116 58
4 58 58
{5: 31, 3: 62, 0: 31}
0 31 31
3 62 31
5 31 31
{2: 60, 4: 40}
4 40 40
2 60 40
{6: 59}
6 59 39
{1: 130, 2: 98, 4: 65}
4 65 65
2 98 65
1 130 65
{7: 108, 6: 81}
6 81 54
7 108 54
{0: 54}
0 54 54
{4: 48, 7: 96}
7 96 48
4 48 48
{3: 96, 0: 48, 1: 96}
1 96 48
0 48 48
3 96 48
{2: 81, 4: 54, 7: 108}
7 108 54
4 54 54
2 81 54
{5: 42, 3: 84}
3 84 42
5 42 42
{1: 82}
1 82 41
{6: 62, 5: 41, 3: 82}
3 82 41
5 41 41
6 62 41
{0: 49, 1: 98}
1 98 49
0 49 49
{7: 98}
7 98 49
{3: 104, 0: 52}
0 52 52
3 104 52
{4: 40, 7: 80, 6: 60}
6 60 40
7 80 40
4 40 40
{5: 55, 3: 110, 0: 55}
0 55 55
3 110 55
5 55 55
{2: 60, 4: 40}
4 40 40
2 60 40
{6: 62}
6 62 41
{

In [27]:
num_jobs = len(jobs_data)
print(num_jobs)

53


In [30]:
count = 0
for i in range(num_jobs):
    for j in range(i+1, num_jobs):
        count = count + 1
print(count)


1378


In [32]:
path_data = compute_k_paths("./input/job_stress_test/53_8.json", k=1)

In [33]:
path_data

{(1, 1): {'paths': [[1]], 'costs': [0]},
 (1, 2): {'paths': [[1, 4, 5, 2]], 'costs': [2]},
 (1, 3): {'paths': [[1, 4, 5, 6, 3]], 'costs': [3]},
 (1, 7): {'paths': [[1, 4, 8, 7]], 'costs': [2]},
 (1, 11): {'paths': [[1, 4, 5, 6, 10, 11]], 'costs': [4]},
 (1, 15): {'paths': [[1, 4, 8, 12, 15]], 'costs': [3]},
 (1, 16): {'paths': [[1, 4, 5, 9, 13, 16]], 'costs': [4]},
 (1, 17): {'paths': [[1, 4, 5, 6, 10, 14, 17]], 'costs': [5]},
 (2, 1): {'paths': [[2, 5, 4, 1]], 'costs': [2]},
 (2, 2): {'paths': [[2]], 'costs': [0]},
 (2, 3): {'paths': [[2, 5, 6, 3]], 'costs': [2]},
 (2, 7): {'paths': [[2, 5, 4, 8, 7]], 'costs': [3]},
 (2, 11): {'paths': [[2, 5, 6, 10, 11]], 'costs': [3]},
 (2, 15): {'paths': [[2, 5, 4, 8, 12, 15]], 'costs': [4]},
 (2, 16): {'paths': [[2, 5, 9, 13, 16]], 'costs': [3]},
 (2, 17): {'paths': [[2, 5, 6, 10, 14, 17]], 'costs': [4]},
 (3, 1): {'paths': [[3, 6, 5, 4, 1]], 'costs': [3]},
 (3, 2): {'paths': [[3, 6, 5, 2]], 'costs': [2]},
 (3, 3): {'paths': [[3]], 'costs': [0]},


In [39]:
print(((1,4) in path_data))

False


In [40]:
if((1,15) in path_data):
    for path in path_data[(1,15)]["paths"]:
        print(path)

[1, 4, 8, 12, 15]


In [41]:
path_nodes = [node_to_idx[x] for x in path]
print(path_nodes)

[0, 8, 11, 14, 5]


In [43]:
print(es_real_to_esidx, node_to_idx)

{1: 0, 2: 1, 3: 2, 7: 3, 11: 4, 15: 5, 16: 6, 17: 7} {1: 0, 2: 1, 3: 2, 7: 3, 11: 4, 15: 5, 16: 6, 17: 7, 4: 8, 5: 9, 6: 10, 8: 11, 9: 12, 10: 13, 12: 14, 13: 15, 14: 16}


In [48]:
def build_routing_options(sender_job, receiver_job):
    
    routing_options = []
    option_counter = 0

    sender_allowed = [
        rid for rid in jobs_data[sender_job]["can_run_on"]
        if rid in es_real_to_esidx
    ]

    receiver_allowed = [
        rid for rid in jobs_data[receiver_job]["can_run_on"]
        if rid in es_real_to_esidx
    ]

    for src_real in sender_allowed:

        for dst_real in receiver_allowed:

            src_es_idx = es_real_to_esidx[src_real]
            dst_es_idx = es_real_to_esidx[dst_real]

            path_key = (src_real, dst_real)

            if path_key not in path_data:
                # an error has to be raised here
                raise ValueError(f"Path not found for nodes: {path_key}")
                continue

            for path in path_data[path_key]["paths"]:

                if not path:
                    continue

                path_nodes = [
                    node_to_idx[x]
                    for x in path
                ]

                routing_options.append(
                    (
                        option_counter,
                        src_es_idx,
                        dst_es_idx,
                        path_nodes
                    )
                )

                option_counter += 1
                
   
    return routing_options

In [51]:
build_routing_options(15, 26)

[(0, 7, 4, [7, 16, 13, 4]),
 (1, 7, 7, [7]),
 (2, 7, 6, [7, 16, 15, 6]),
 (3, 6, 4, [6, 15, 12, 13, 4]),
 (4, 6, 7, [6, 15, 16, 7]),
 (5, 6, 6, [6])]

In [53]:
for i in range(3):
    for j in range(i+1, 3):
        print(i,j)
    

0 1
0 2
1 2


In [54]:
path_nodes = [7, 16, 13, 4]

In [55]:
num_hops = len(path_nodes) - 1
for h in range(num_hops):
    print(min(path_nodes[h], path_nodes[h+1]), max(path_nodes[h], path_nodes[h+1]))

7 16
13 16
4 13
